In [ ]:
# -*- coding: utf-8 -*-

# GeoSolvrFINAL

Automatically generated from the uploaded Python script.

Original Colab note preserved from the source file.


In [ ]:
!pip install google.cloud
!pip install openai-agents nest_asyncio --quiet
!pip install google-cloud-vision

from agents import Agent, function_tool, Runner, WebSearchTool, ModelSettings
from openai import OpenAI
from google.colab import userdata  # Imports the userdata function to access secrets
import base64 # Import the base64 module
from pydantic import BaseModel, Field
import os
import nest_asyncio
from openai import OpenAI
import re
from agents import Agent, function_tool, Runner, WebSearchTool, ModelSettings

from google.cloud import vision

## Image Analysis


In [ ]:
api = userdata.get('OPENAI_API_KEY')  # Sets the API key to my secret key
client = OpenAI(api_key=api)  # Initializes the client with the API key


image = input("Enter the image URL: ")#makes a variable with the image url

def imageAnalysis(image):
  response= client.responses.create(
    model="gpt-4.1",#choses model
    tools=[{
           #"type": "cnn"
           "type": "web_search_preview"}],  # Enables web search tool
    input= #the prompt for the model
         [
        {
            "role": "user",
            "content": [
                { "type": "input_text", "text": '''Analyze this Google Street View image and identify key visual features.
            Specifically, describe the landscape (urban/rural, arid/tropical, flat/mountainous), infrastructure
            (signs, poles, guardrails), architecture styles, and weather conditions.
            Provide the analysis clearly in bullet points, the image is encoded in bytes below: ''' },
                {
                "type": "input_image",
                "image_url": f"{image}",
                },
            ],
        },
         ])
  return response.output_text  # Returns the AI's response

imageAnalysisOutput = imageAnalysis(image)  # Prints the AI's response

## Language Analysis


In [ ]:
api = userdata.get('OPENAI_API_KEY')  # Sets the API key to my secret key
client = OpenAI(api_key=api)  # Initializes the client with the API key

os.environ['OPENAI_API_KEY'] = api


# Set the environment variable to your service account file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = '/content/credentials.json'#setting the api key

vision_client = vision.ImageAnnotatorClient() #sets vision client
nest_asyncio.apply()




@function_tool(name_override='OCR', description_override='Detect any text in the given image')#defines tool
def OCR(image:str)->str:
  '''
    image: path to image file (e.g., 'my_folder/image.jpg')
    returns: detected text as string
  '''
  try:
    with open(image, 'rb') as image_file:    #tries to read the image in binary and sets it the content variable
      content = image_file.read()

          # Construct image object for Google Cloud
    image_obj = vision.Image(content=content)

          # Run text detection
    response = vision_client.text_detection(image=image_obj)
    annotations = response.text_annotations

          # Check for errors
    if response.error.message:
      return f"OCR Error: {response.error.message}"

    if not annotations:
      return "No text detected in the image."

          # The first item is the full detected text
    return annotations[0].description.strip()
  except Exception as e:
    return f"An error occurred: {str(e)}"




languageAnalysis = Agent(#defines the language analysis agents
    name= 'language analysis', # gives it a name
    instructions= ("Analyze any visible text or languages in this Google Street View image. "
            "Identify the language and correlate them to specific countries or regions. "
            "Include license plate details, road signs, and markings if visible. "
            "Provide your analysis in bullet points clearly"),    #b
    model = 'gpt-4.1',
    tools= [OCR, WebSearchTool()],
    model_settings = ModelSettings(tool_choice='auto'),
  )



result = Runner.run_sync(languageAnalysis, f'Analyze any visible text or languages in this Google Street View image: {image}')


languageAnalysisOutput = result # stores the AI's response

## Map Analysis


In [ ]:
api = userdata.get('OPENAI_API_KEY')  # Sets the API key to my secret key
client = OpenAI(api_key=api)  # Initializes the client with the API key

def mapReasoning(languageAnalysisOutput, imageAnalysisOutput, image):
  response = client.responses.create(
    model="gpt-4.1",
    input=(
            "Based on the following visual and language analysis data, correlate possible locations on a world map. "
            "Use geographic databases, terrain, and country APIs to eliminate unlikely locations and narrow down potential countries. "
            "Provide a clear, concise list of candidate locations with reasoning:"
            f"Visual Analysis:{imageAnalysisOutput}"
            f"Language Analysis:{languageAnalysisOutput}"
            f"Image:{image}"
    ),
    #tools=[{
           #"type": "geographic_databases"
           #"type": "country_api"
           #"type": "terrain_api"}],  # Enables web search tool
  )
  return response.output_text  # Returns the AI's response

mapReasoningOutput = mapReasoning(languageAnalysisOutput, imageAnalysisOutput, image)  # Prints the AI's response

## Decision Making


In [ ]:
api = userdata.get('OPENAI_API_KEY')  # Sets the API key to my secret key
client = OpenAI(api_key=api)  # Initializes the client with the API key

#   return response.output_text  # Returns the AI's response

# class FinalDecisionResponse(BaseModel):
#     final_country: str
#     confidence_score: int
#     reasoning: str

# decisionMakingOutput = decisionMaking(image, mapReasoningOutput, imageAnalysisOutput, languageAnalysisOutput)  # saves the AI's response


class FinalDecisionResponse(BaseModel):
     final_city: str
     confidence_score: int
     reasoning: str



decisionMaking = Agent(
    name="Decision Making",
    instructions="Aggregate and evaluate data from visual analysis, language analysis, and map reasoning. "
             "Weigh each agent's input based on confidence and ensemble strategies. "
             "Make a final guess identifying the most probable country, city or subdivision. Provide your decision clearly with reasoning and a confidence score out of 100:"
             f"Visual Analysis:{imageAnalysisOutput}"
             f"Language Analysis:{result}"
             f"Map Reasoning:{mapReasoningOutput}"
             f"Image:{image}",
    model="gpt-4.1",
    output_type=FinalDecisionResponse,
)


decisionMakingOutput = Runner.run_sync(decisionMaking, "Aggregate and evaluate data from visual analysis, language analysis, and map reasoning to make a final decision on the area where the image if from")
print(decisionMakingOutput.final_output)

## Final Testing


In [ ]:
outputs= []
image = input("Enter the image URL: ")

for i in range(3):
  imageAnalysisOutput = imageAnalysis(image)
  result = Runner.run_sync(languageAnalysis, f'Analyze any visible text or languages in this Google Street View image: {image}')
  mapReasoningOutput = mapReasoning(result, imageAnalysisOutput, image)
  decisionMakingOutput = Runner.run_sync(decisionMaking, f"Aggregate and evaluate data from visual analysis: {imageAnalysisOutput}, language analysis: {result}, and map reasoning: {mapReasoning} to make a final decision on the area where the image if from")
  print(f'OUTPUT {i+1}')
  print(decisionMakingOutput.final_output)
  outputs.append(decisionMakingOutput.final_output)

## Evaluation


In [ ]:
estimated_confidence_levels = []
final_confidence = []

yn_agent = Agent(
    name="Mr. YN",
    instructions="You will either answer with 'y' or 'n' ONLY, eg. q: 'does 2 + 2 = 4?' a: 'y'",
    model="o3-mini",
)

confidence_agent = Agent(
    name="Mr. confidence sniffer",
    instructions="evaluate the reasoning presented to you from 1 to 100. Only write your response (the evaluation from 1-100). Base this purely on how sound you feel the logic is. eg. q: 'I thought this country was france since the sky looked like it' a: '32'",
    model="o3-mini",
)

# Helper Functions
def test_similiarity(x, y):
    result = Runner.run_sync(
        yn_agent,
        f"rAre {x} and {y} basically the same area?"
    )
    return result.final_output.lower() == "y"

def majority_vote(votes):
    if test_similiarity(votes[0], votes[1]):
        return votes[0]
    elif test_similiarity(votes[1], votes[2]):
        return votes[1]
    elif test_similiarity(votes[0], votes[2]):
        return votes[2]
    else:
        print("Tie detected!")
        return "TIE"

def evaluate_confidence(reasoning_list):
    for reason in reasoning_list:
        result = Runner.run_sync(confidence_agent, reason)
        print(f"Confidence agent rated: {result.final_output}")
        estimated_confidence_levels.append(int(result.final_output))

def final_guess(guess):
    print(f"Final Guess: {guess}")

# Extract details
guesses = [out.final_city for out in outputs]
confidences = [out.confidence_score for out in outputs]
reasonings = [out.reasoning for out in outputs]

vote_result = majority_vote(guesses)

if vote_result != "TIE":
    final_guess(vote_result)
else:
    print("No majority — falling back to confidence evaluation")
    evaluate_confidence(reasonings)

    # Combine reported confidence and estimated confidence
    for i in range(3):
        combined = (confidences[i]*100 + estimated_confidence_levels[i]) / 2
        final_confidence.append(combined)
        print(f"Combined confidence for agent {i+1}: {combined}")

    # Choose most confident agent
    best_index = final_confidence.index(max(final_confidence))
    final_guess(guesses[best_index])

## Google Maps URL Transformer

Transformer that converts a Google Maps location link into a Street View Static API URL before passing it into the multi-agent system.


In [ ]:
import re

def extract_values(gmaps_url):
# Extract panoId from the Google Maps link
    pano_match = re.search(r'!1s([a-zA-Z0-9_-]{22})', gmaps_url)
    pano_id = pano_match.group(1) if pano_match else None

# Extract the heading
    yaw_match = re.search(r'yaw%3D([\d\.]+)', gmaps_url)
    heading = yaw_match.group(1) if yaw_match else "0"

# Extract pitch
    pitch_match = re.search(r'pitch%3D([-?\d\.]+)', gmaps_url)
    pitch = pitch_match.group(1) if pitch_match else "0"

    return pano_id, heading, pitch

def build_sv_url(pano_id, heading, pitch, api_key):
    '''
A function that builds the URL for the Google Street View Static API given what it extracted and the input API key
    '''
    base_url = "https://maps.googleapis.com/maps/api/streetview"
    return f"{base_url}?pano={pano_id}&heading={heading}&pitch={pitch}&fov=90&size=640x640&key={api_key}"

# Ask user for input here
gmaps_url = input("Paste your Google Maps Street View URL: ").strip()
api_key = input("Enter your Google API key: ").strip()

pano_id, heading, pitch = extract_values(gmaps_url)

# Validate official coverage, return the url that we can use to call the API

if pano_id:
    sv_url = build_sv_url(pano_id, heading, pitch, api_key)
    print("\n Street View Static API URL:")
    print(sv_url)
else:
    print(" Could not extract panoId from the URL.")